## Sampling with logprobs, temperature, top-p, top-k

In [4]:
from dotenv import load_dotenv

load_dotenv()

True

In [5]:
import os
from openai import OpenAI

client = OpenAI(
    base_url=os.environ["AZURE_OPENAI_ENDPOINT"] + "/openai/v1/",
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
)

response = client.chat.completions.create(   
    model="gpt-4.1",
    messages=[
        {"role": "system", "content": "Answer in short sentences."},
        {"role": "user", "content": "Adam is holding an apple. So who'll eat the apple?"}
    ],
    logprobs=True,
    top_logprobs=3,
    temperature=0.0,
    seed=42,
)

print(response.choices[0].message.content)

Based on the information, Adam is most likely to eat the apple.


### Logprobs

https://developers.openai.com/cookbook/examples/using_logprobs

logprob is `log(p)`, where p = probability of a token occurring at a specific position based on the previous tokens in the context. To recover the probability: `p=e^logprob`.

```
logit
   ↓ softmax
probability
   ↓ log
log probability
```

- Logprob can be any negative number or 0. 0 corresponds to 100% probability (p=1).
- Higher log probabilities suggest a higher likelihood of the token in that context. This allows users to gauge the model’s confidence in its output or explore alternative responses the model considered.

#### Why logprobs

Computing joint probability of a sequence requires multiplying the probabilities. e.g. Given the probabilities:

```
The       → 0.90
 cat      → 0.80
 is       → 0.95
 sleeping → 0.60
```

The joint probability of sequence `The cat is sleeping` is `0.90×0.80×0.95×0.60 = 0.4104`.

Now, using logprobs instead...

```
log(0.90) = -0.105
log(0.80) = -0.223
log(0.95) = -0.051
log(0.60) = -0.511
```

since `log(ab)=log(a)+log(b)`, we can add the logprobs: `−0.105−0.223−0.051−0.511 = −0.890`. Exponentiating that sum gives the joint probability: `e^−0.890 = 0.410`.

Logprobs allow us to compute the joint log probability of a sequence by summing the logprobs of its tokens which is faster and numerically stable than multiplication of small probabilities.

In [6]:
import math

print("Token\t\tLog Prob\tProbability\tTop-K Alternatives", end="\n\n")
for item in response.choices[0].logprobs.content:
    prob = math.exp(item.logprob)
    top_k = " | ".join(
        f"{t.token!r} ({math.exp(t.logprob):.2f})" for t in item.top_logprobs
    )
    print(f"{item.token}\t\t{item.logprob:.4f}\t\t{prob:.2f}\t\t{top_k}")

Token		Log Prob	Probability	Top-K Alternatives

Based		-0.0570		0.94		'Based' (0.94) | 'The' (0.02) | 'From' (0.02)
 on		-0.0000		1.00		' on' (1.00) | ' only' (0.00) | ' solely' (0.00)
 the		-0.0039		1.00		' the' (1.00) | ' your' (0.00) | ' what' (0.00)
 information		-0.1513		0.86		' information' (0.86) | ' sentence' (0.07) | ' statement' (0.06)
,		-0.0340		0.97		',' (0.97) | ' given' (0.03) | ' provided' (0.00)
 Adam		-0.0592		0.94		' Adam' (0.94) | ' it' (0.03) | " it's" (0.01)
 is		-0.0184		0.98		' is' (0.98) | ' might' (0.02) | ' will' (0.00)
 most		-0.5380		0.58		' most' (0.58) | ' likely' (0.40) | ' holding' (0.01)
 likely		0.0000		1.00		' likely' (1.00) | 'likely' (0.00) | ' probably' (0.00)
 to		-0.0032		1.00		' to' (1.00) | ' the' (0.00) | ' going' (0.00)
 eat		0.0000		1.00		' eat' (1.00) | ' be' (0.00) | 'eat' (0.00)
 the		-0.0000		1.00		' the' (1.00) | ' it' (0.00) | 'the' (0.00)
 apple		-0.0000		1.00		' apple' (1.00) | ' Apple' (0.00) | 'apple' (0.00)
.		-0.1805		0.83		'.' 

### Temperature

https://github.com/karpathy/nanochat/blob/master/nanochat/gpt.py

Temperature is the single best knob for global randomness. Set it to: 0.0–0.5 for deterministic/precise, 0.7–1.0 for creative, >1.0 for random outputs.

Temperature controls entropy of the token distribution. Temperature rescales logits before softmax using: `logit / T`.

- `T = 0.0`: The top token is the most dominant.
- `T = 0.5`: Sharper distribution. The top token becomes more dominant.
- `T = 1.0`: No scaling. The default.
- `T = 2.0`: Flatter distribution. Lower tokens have higher chance.


### Top K and Top P

top_k and top_p are like filters on the probability distribution before sampling. They control how many candidate tokens the model is allowed to choose from. `top_k` gives a fixed number of choices; `top_p` gives a variable-size set that covers most of the mass.

`top_k = K` keeps only the K tokens with highest probability. It sets others to 0 probability and renormalizes remaining probabilities.

### Top P (nucleus) sampling


`top_p = P` sorts tokens by probability descending, takes the smallest prefix whose cumulative probability ≥ p, keeps those tokens, zeroes the rest, renormalizes.

### Pseudo Code for next token generation

```python
import math
import random


def softmax(logits_dict):
    values = list(logits_dict.values())

    # convert logits to exponentiated values
    max_logit = max(values)
    exp_vals = {k: math.exp(v - max_logit) for k, v in logits_dict.items()}

    # convert logits to probabilities
    total = sum(exp_vals.values())
    return {k: v / total for k, v in exp_vals.items()}

def filter_top_k(probs, k):
    # sort tokens by probability descending
    sorted_tokens = sorted(probs.items(), key=lambda x: x[1], reverse=True)

    # keep only the k highest probability tokens
    kept = dict(sorted_tokens[:k])

    # renormalize so probabilities sum to 1 again
    total = sum(kept.values())
    return {k: v / total for k, v in kept.items()}

def filter_top_p(probs, p):
    # sort tokens by probability
    sorted_tokens = sorted(probs.items(), key=lambda x: x[1], reverse=True)

    cumulative = 0
    kept = {}

    for token, prob in sorted_tokens:
        cumulative += prob
        kept[token] = prob

        # stop when cumulative probability crosses threshold
        if cumulative >= p:
            break

    # renormalize remaining tokens
    total = sum(kept.values())
    return {k: v / total for k, v in kept.items()}

def sample_from_distribution(probs):
    tokens = list(probs.keys())
    weights = list(probs.values())

    # randomly choose token according to probability weights
    return random.choices(tokens, weights=weights, k=1)[0]


# Generation pipeline

# logits: array[vocab_size]
# counts: dict[token] -> count so far
# params: temperature, top_k, top_p, freq_penalty, presence_penalty, repetition_penalty, do_sample

# apply penalties to logits
for token in vocab:
    if counts[token] > 0:
        # penalize tokens that already appeared
        logits[token] -= presence_penalty

        # additional penalty proportional to count
        logits[token] -= freq_penalty * counts[token]

    if repetition_penalty > 1.0 and counts[token] > 0:
        # multiplicative reduction in logit
        logits[token] = logits[token] / repetition_penalty

# temperature scaling
# higher temp → logits shrink → probabilities flatten
scaled_logits = {k: v / temperature for k, v in logits.items()}

# logits → probabilities
probs = softmax(scaled_logits)

# restrict candidate tokens
probs = filter_top_k(probs, top_k)
probs = filter_top_p(probs, top_p)

# choose token
if do_sample:
    # probabilistic sampling
    next_token = sample_from_distribution(probs)
else:
    # greedy selection
    next_token = max(probs, key=probs.get)

print("Final probabilities:", probs)
print("Selected token:", next_token)
```